# R3maJ — Colab Training + Native W&B

W&B authentication is loaded from the Colab Secret and passed to the R3maJ C++ core. The notebook does **not** call `wandb.login()` and does not duplicate the C++ telemetry.

Add `WANDB_API_KEY` under **Colab → Secrets** before running.

In [ ]:
# 1. Secure W&B environment — NO wandb.login()
import os
from google.colab import userdata
WANDB_API_KEY = userdata.get('WANDB_API_KEY')
if not WANDB_API_KEY:
    raise RuntimeError('WANDB_API_KEY not found. Add it under Colab -> Secrets.')
WANDB_API_KEY = WANDB_API_KEY.strip()
if not WANDB_API_KEY:
    raise RuntimeError('WANDB_API_KEY is empty.')
os.environ['WANDB_API_KEY'] = WANDB_API_KEY
os.environ['WANDB_MODE'] = 'online'
print('W&B API key loaded from Colab Secrets.')
print('Native C++ W&B telemetry enabled through WANDB_API_KEY.')

In [ ]:
# 2. Clone/update repository
import os, subprocess
ROOT='/content/R3maJ'
REPO='https://github.com/vfxjamer/R3maJ.git'
if not os.path.isdir(os.path.join(ROOT,'.git')):
    subprocess.run(['git','clone','--depth','1',REPO,ROOT],check=True)
else:
    subprocess.run(['git','-C',ROOT,'pull'],check=False)
print('ROOT:',ROOT)

In [ ]:
# 3. Google Drive restore
import os, shutil, glob
from google.colab import drive
if not os.path.isdir('/content/drive'):
    drive.mount('/content/drive')
DRIVE_CKPT='/content/drive/MyDrive/R3maJ/checkpoints'
LOCAL_CKPT='/content/R3maJ/build/checkpoints'
DRIVE_REPLAY='/content/drive/MyDrive/R3maJ/serialized_replays.bin'
LOCAL_REPLAY='/content/R3maJ/build/serialized_replays.bin'
os.makedirs(LOCAL_CKPT,exist_ok=True)
os.makedirs(DRIVE_CKPT,exist_ok=True)
def ts_of(d):
    try:return int(os.path.basename(d))
    except:return -1
dirs=sorted([d for d in glob.glob(os.path.join(DRIVE_CKPT,'*')) if os.path.isdir(d)],key=ts_of)
if dirs:
    newest=dirs[-1];dest=os.path.join(LOCAL_CKPT,os.path.basename(newest))
    if not os.path.isdir(dest):
        print('Restoring checkpoint:',os.path.basename(newest));shutil.copytree(newest,dest)
if os.path.exists(DRIVE_REPLAY) and not os.path.exists(LOCAL_REPLAY):
    print('Copying replay binary');shutil.copy(DRIVE_REPLAY,LOCAL_REPLAY)
print('Checkpoints:',len(dirs),'Replay:',os.path.exists(LOCAL_REPLAY))

In [ ]:
# 4. Curriculum + native telemetry sanity check
import os
pm=os.path.join(ROOT,'src','PhaseManager.cpp')
main=os.path.join(ROOT,'src','main.cpp')
checks=[(pm,'MECH_RAMP_END','mechanical ramp'),(pm,'PhaseManager::PhaseManager()','phase table'),(main,'GetRewards(g_totalTimesteps)','auto phase selection'),(main,'WANDB_API_KEY','native W&B support')]
for path,needle,label in checks:
    ok=os.path.exists(path) and needle in open(path,errors='ignore').read()
    print(('OK  ' if ok else 'MISS ')+label)
if not all(os.path.exists(p) and n in open(p,errors='ignore').read() for p,n,_ in checks):
    raise RuntimeError('Core sanity check failed.')

In [ ]:
# 5. Install dependencies and inspect GPU
import subprocess, torch
subprocess.run(['apt-get','update','-qq'],check=True)
subprocess.run(['apt-get','install','-y','-qq','build-essential','cmake','git','libpython3-dev','pkg-config'],check=True)
print('Python:',__import__('sys').version.split()[0])
print('Torch:',torch.__version__)
print('CUDA:',torch.cuda.is_available())
if torch.cuda.is_available():print('GPU:',torch.cuda.get_device_name(0))

In [ ]:
# 6. Configure and build
import os, subprocess, torch
os.chdir(ROOT)
torch_prefix=os.path.dirname(torch.__file__)
cfg=subprocess.run(['cmake','-S','.','-B','build','-DCMAKE_BUILD_TYPE=Release',f'-DTORCH_INSTALL_PREFIX={torch_prefix}'],capture_output=True,text=True)
print((cfg.stdout+cfg.stderr)[-5000:])
if cfg.returncode!=0:raise RuntimeError('CMake configure failed')
build=subprocess.run(['cmake','--build','build','-j',str(os.cpu_count() or 2)],capture_output=True,text=True)
print((build.stdout+build.stderr)[-5000:])
if build.returncode!=0:raise RuntimeError('Build failed')
print('Binary:',os.path.exists(os.path.join(ROOT,'build','R3maJ')))

In [ ]:
# 7. 24/7 checkpoint backup daemon
import os, glob, shutil, threading, time
BUILD=os.path.join(ROOT,'build')
LOCAL_CKPT=os.path.join(BUILD,'checkpoints')
DRIVE_CKPT='/content/drive/MyDrive/R3maJ/checkpoints'
os.makedirs(LOCAL_CKPT,exist_ok=True);os.makedirs(DRIVE_CKPT,exist_ok=True)
def backup_once():
    for d in glob.glob(os.path.join(LOCAL_CKPT,'*')):
        if not os.path.isdir(d):continue
        name=os.path.basename(d);dest=os.path.join(DRIVE_CKPT,name)
        if os.path.exists(dest):continue
        tmp=dest+'.tmp'
        try:shutil.copytree(d,tmp);shutil.move(tmp,dest);print('[backup]',name,'uploaded',flush=True)
        except Exception as e:print('[backup] error:',e,flush=True);shutil.rmtree(tmp,ignore_errors=True)
def backup_loop():
    while True:
        backup_once();time.sleep(60)
if '_R3MAJ_BACKUP_DAEMON_' not in globals():
    globals()['_R3MAJ_BACKUP_DAEMON_']=True
    threading.Thread(target=backup_loop,daemon=True).start()
print('Backup daemon running.')

In [ ]:
# 8. Start R3maJ — native C++ W&B telemetry
import os, subprocess, torch
os.chdir(BUILD)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
REPLAY_ARG=['--replays','serialized_replays.bin'] if os.path.exists('serialized_replays.bin') else []
CMD=['stdbuf','-oL','-eL','./R3maJ','--device',DEVICE,'--phase','-1','--save-dir','checkpoints','--games','164']+REPLAY_ARG
print('BASE CMD:',' '.join(CMD))
print('W&B metrics are emitted by src/main.cpp. No Python W&B logger is used.')
with open('train.log','a') as log:
    proc=subprocess.Popen(CMD,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
    rc=proc.wait()
print('R3maJ exit code:',rc)

In [ ]:
# 9. STATUS
import os,subprocess,glob
print('Drive mounted:',os.path.isdir('/content/drive'))
print('Local checkpoints:',sorted(os.path.basename(d) for d in glob.glob('/content/R3maJ/build/checkpoints/*')))
r=subprocess.run(['pgrep','-af','R3maJ'],capture_output=True,text=True)
print('R3maJ processes:',r.stdout.strip() or 'none')

In [ ]:
# 10. STOP cleanly
import subprocess,time
r=subprocess.run(['pkill','-TERM','-f','R3maJ'],capture_output=True,text=True)
print('SIGTERM sent:',r.returncode==0)
time.sleep(20)
r2=subprocess.run(['pgrep','-af','R3maJ'],capture_output=True,text=True)
if r2.stdout.strip():
    print('Force killing remaining R3maJ processes');subprocess.run(['pkill','-9','-f','R3maJ'],capture_output=True)
print('Remaining:',subprocess.run(['pgrep','-af','R3maJ'],capture_output=True,text=True).stdout.strip() or 'none')